# Supply Chain Data Ingestion

Ingests CSV source data from the workspace `Source_Data` folder into Delta tables in Unity Catalog (bronze layer). Reads 9 CSV files covering financial transactions, item master, MRP, CDP pricing, supplier master/terms, market data, revenue units, and margin units.

In [0]:
# ---------------------------------------------------------------------------
# Configuration: catalog, schema, and source data base path
# ---------------------------------------------------------------------------
CATALOG = "workspace"
SCHEMA = "supply_chain_bronze"
BASE_PATH = "/Workspace/Users/tushar.kulkarni89@gmail.com/Supply_Chain_Project/Source_Data"

# Create the schema (catalog already exists as `workspace`)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Using catalog.schema: {CATALOG}.{SCHEMA}")

In [0]:
# ---------------------------------------------------------------------------
# Ingest all 9 CSV source files into Delta tables (bronze layer)
# ---------------------------------------------------------------------------
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

# Each entry: (subfolder, filename, table_name, schema, table_comment)
source_files = [
    (
        "Financial",
        "financial_transactions.csv",
        "financial_transactions",
        StructType([
            StructField("transaction_id", StringType(), True),
            StructField("transaction_date", DateType(), True),
            StructField("supplier_id", StringType(), True),
            StructField("item_id", StringType(), True),
            StructField("region", StringType(), True),
            StructField("quantity", IntegerType(), True),
            StructField("revenue", DoubleType(), True),
            StructField("cost", DoubleType(), True),
            StructField("rebate_pct", DoubleType(), True),
            StructField("currency", StringType(), True),
        ]),
        "Financial transactions: supplier-item-level purchase records with revenue, cost, and rebate details.",
    ),
    (
        "Item_master",
        "item_master.csv",
        "item_master",
        StructType([
            StructField("item_id", StringType(), True),
            StructField("item_description", StringType(), True),
            StructField("category", StringType(), True),
            StructField("sub_category", StringType(), True),
            StructField("brand", StringType(), True),
            StructField("uom", StringType(), True),
            StructField("standard_cost", DoubleType(), True),
        ]),
        "Item master: product catalog with category, brand, unit of measure, and standard cost.",
    ),
    (
        "mrp",
        "mrp.csv",
        "mrp",
        StructType([
            StructField("item_id", StringType(), True),
            StructField("mrp", DoubleType(), True),
            StructField("effective_date", DateType(), True),
            StructField("currency", StringType(), True),
            StructField("region", StringType(), True),
        ]),
        "MRP: Maximum Retail Price per item by region and effective date.",
    ),
    (
        "cdp_price",
        "cdp_price.csv",
        "cdp_price",
        StructType([
            StructField("item_id", StringType(), True),
            StructField("supplier_id", StringType(), True),
            StructField("cdp_price", DoubleType(), True),
            StructField("effective_date", DateType(), True),
            StructField("currency", StringType(), True),
        ]),
        "CDP Price: Cost Delivered Price per supplier-item combination with effective date.",
    ),
    (
        "supplier",
        "supplier_master.csv",
        "supplier_master",
        StructType([
            StructField("supplier_id", StringType(), True),
            StructField("supplier_name", StringType(), True),
            StructField("supplier_region", StringType(), True),
            StructField("supplier_type", StringType(), True),
            StructField("capacity_units", IntegerType(), True),
            StructField("lead_time_days", IntegerType(), True),
            StructField("quality_score", DoubleType(), True),
        ]),
        "Supplier Master: supplier profile with region, type, capacity, lead time, and quality score.",
    ),
    (
        "supplier",
        "supplier_terms.csv",
        "supplier_terms",
        StructType([
            StructField("supplier_id", StringType(), True),
            StructField("item_id", StringType(), True),
            StructField("allocated_capacity_units", IntegerType(), True),
            StructField("minimum_order_units", IntegerType(), True),
            StructField("target_rebate_pct", DoubleType(), True),
            StructField("contract_status", StringType(), True),
        ]),
        "Supplier Terms: supplier-item contract terms including allocated capacity, minimum order, and target rebate.",
    ),
    (
        "market",
        "market_data_idc.csv",
        "market_data_idc",
        StructType([
            StructField("market_id", StringType(), True),
            StructField("item_id", StringType(), True),
            StructField("region", StringType(), True),
            StructField("market_demand", IntegerType(), True),
            StructField("market_price", DoubleType(), True),
            StructField("market_share", DoubleType(), True),
            StructField("market_date", DateType(), True),
        ]),
        "Market Data IDC: market demand, price, and share per item-region at a point in time.",
    ),
    (
        "revenue",
        "revenue_units.csv",
        "revenue_units",
        StructType([
            StructField("item_id", StringType(), True),
            StructField("region", StringType(), True),
            StructField("revenue_units", IntegerType(), True),
            StructField("period", DateType(), True),
        ]),
        "Revenue Units: monthly revenue units per item-region.",
    ),
    (
        "margin",
        "margin_units.csv",
        "margin_units",
        StructType([
            StructField("item_id", StringType(), True),
            StructField("region", StringType(), True),
            StructField("margin_units", IntegerType(), True),
            StructField("period", DateType(), True),
        ]),
        "Margin Units: monthly margin units per item-region.",
    ),
]

results = []

for folder, filename, table_name, schema, comment in source_files:
    file_path = f"{BASE_PATH}/{folder}/{filename}"
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    print(f"\nIngesting: {folder}/{filename}  -->  {full_table}")

    # Read CSV with explicit schema
    df = spark.read.csv(file_path, header=True, schema=schema, dateFormat="yyyy-MM-dd")

    row_count = df.count()

    # Write as Delta table (overwrite mode for idempotent re-runs)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table)

    # Add table comment
    spark.sql(f"ALTER TABLE {full_table} SET TBLPROPERTIES (comment = '{comment}')")

    results.append((table_name, row_count))
    print(f"  Rows ingested: {row_count:,}")

print("\n============================================")
print("Ingestion complete!")
print("============================================")

In [0]:
# ---------------------------------------------------------------------------
# Validation: row counts and schema for each ingested table
# ---------------------------------------------------------------------------
print(f"{'Table':<25} {'Rows':>10}")
print("-" * 37)

for table_name, row_count in results:
    print(f"{table_name:<25} {row_count:>10,}")

print("-" * 37)

# Show a sample from each table
for table_name, _ in results:
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    print(f"\n--- {full_table} (sample) ---")
    display(spark.table(full_table).limit(5))